# Session 05 Practice: Hierarchical State Machines (HSM) Behaviour Planner
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 3 | Session 5**  

---

### 🎯 Learning Objectives
1. Implement a **Rule-Based Hierarchical State Machine (HSM)** for autonomous intersection & stop-sign handling.
2. Structure discrete driving states (`FOLLOW_LANE`, `DECELERATE_TO_STOP`, `STAY_STOPPED`, `CROSS_INTERSECTION`).
3. Define strict guard condition transitions based on distance to stop line, vehicle speed, and perception clearances.
4. Visualize state transition timelines, velocity profiles, and phase portraits using **Plotly**.

---

### 📐 Mathematical Formulation

State transitions are governed by predicate logic:
1. $\text{FOLLOW\_LANE} \xrightarrow{d_{\text{stop}} \le d_{\text{thresh}}} \text{DECELERATE\_TO\_STOP}$
2. $\text{DECELERATE\_TO\_STOP} \xrightarrow{d_{\text{stop}} \le 1.5\text{m} \land v < 0.2\text{m/s}} \text{STAY\_STOPPED}$
3. $\text{STAY\_STOPPED} \xrightarrow{t_{\text{wait}} \ge 3.0\text{s} \land \text{clearance}=\text{True}} \text{CROSS\_INTERSECTION}$
4. $\text{CROSS\_INTERSECTION} \xrightarrow{s > s_{\text{stop}} + 15\text{m}} \text{FOLLOW\_LANE}$


In [ ]:
from enum import Enum
import math
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"


## 1. HSM Behaviour Planner Implementation

In [ ]:
class State(Enum):
    FOLLOW_LANE = "FOLLOW_LANE"
    DECELERATE_TO_STOP = "DECELERATE_TO_STOP"
    STAY_STOPPED = "STAY_STOPPED"
    CROSS_INTERSECTION = "CROSS_INTERSECTION"

class HSMBehaviourPlanner:
    def __init__(self, stop_wait_time: float = 3.0, stop_dist_threshold: float = 25.0):
        self.state = State.FOLLOW_LANE
        self.stop_wait_time = stop_wait_time
        self.stop_dist_threshold = stop_dist_threshold
        self.stop_timer = 0.0
        self.target_speed = 12.0 # Cruising speed m/s (~43 km/h)

    def update(self, ego_s: float, ego_v: float, stop_sign_s: float,
               is_intersection_clear: bool, dt: float) -> tuple[State, float]:
        dist_to_stop = stop_sign_s - ego_s if stop_sign_s is not None else float('inf')
        
        if self.state == State.FOLLOW_LANE:
            self.target_speed = 12.0
            if 0 < dist_to_stop <= self.stop_dist_threshold:
                self.state = State.DECELERATE_TO_STOP

        elif self.state == State.DECELERATE_TO_STOP:
            self.target_speed = max(0.0, min(12.0, (dist_to_stop / self.stop_dist_threshold) * 12.0))
            if dist_to_stop <= 1.5 and ego_v < 0.2:
                self.state = State.STAY_STOPPED
                self.stop_timer = 0.0

        elif self.state == State.STAY_STOPPED:
            self.target_speed = 0.0
            self.stop_timer += dt
            if self.stop_timer >= self.stop_wait_time and is_intersection_clear:
                self.state = State.CROSS_INTERSECTION

        elif self.state == State.CROSS_INTERSECTION:
            self.target_speed = 8.0 # moderate speed through turn/intersection
            if ego_s > stop_sign_s + 15.0: # passed intersection
                self.state = State.FOLLOW_LANE
                self.stop_timer = 0.0

        return self.state, self.target_speed


## 2. Stop-Sign Encounter Simulation

In [ ]:
def simulate_scenario(dt: float = 0.1, total_time: float = 16.0) -> pl.DataFrame:
    planner = HSMBehaviourPlanner(stop_wait_time=3.0, stop_dist_threshold=25.0)
    
    stop_sign_s = 40.0
    ego_s = 0.0
    ego_v = 12.0
    
    logs = []
    num_steps = int(total_time / dt)
    
    for step in range(num_steps):
        t = step * dt
        # Intersection is clear after t = 7.0s
        is_clear = (t > 7.0)
        
        state, target_v = planner.update(ego_s, ego_v, stop_sign_s, is_clear, dt)
        
        # Simple longitudinal controller towards target_v
        accel = (target_v - ego_v) * 1.5
        accel = max(-3.0, min(2.0, accel))
        ego_v = max(0.0, ego_v + accel * dt)
        ego_s += ego_v * dt
        
        logs.append({
            "time_s": round(t, 2),
            "ego_s_m": round(ego_s, 2),
            "ego_v_mps": round(ego_v, 2),
            "target_v_mps": round(target_v, 2),
            "state": state.value,
            "stop_timer_s": round(planner.stop_timer, 2),
            "dist_to_stop_m": round(max(0.0, stop_sign_s - ego_s), 2)
        })
        
    return pl.DataFrame(logs)

df_sim = simulate_scenario(dt=0.1, total_time=16.0)
print(f"Simulation finished: {len(df_sim)} steps logged.")


## 3. State Machine Visualizations & Dynamic Telemetry Plots

In [ ]:
# 1. Multi-Channel Timeline Plot
fig_tl = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Velocity Evolution & Target Command", "Distance to Stop Line"),
    vertical_spacing=0.1
)

fig_tl.add_trace(
    go.Scatter(x=df_sim["time_s"], y=df_sim["ego_v_mps"], mode="lines",
               line=dict(color="#1f77b4", width=3), name="Ego Speed v(t)"),
    row=1, col=1
)
fig_tl.add_trace(
    go.Scatter(x=df_sim["time_s"], y=df_sim["target_v_mps"], mode="lines",
               line=dict(color="#2ca02c", width=2, dash="dash"), name="Target Speed v_cmd"),
    row=1, col=1
)

fig_tl.add_trace(
    go.Scatter(x=df_sim["time_s"], y=df_sim["dist_to_stop_m"], mode="lines",
               line=dict(color="#ff7f0e", width=2.5), name="Dist to Stop Line (m)"),
    row=2, col=1
)

fig_tl.update_layout(
    title="HSM Behaviour Planner: Stop-Sign Encounter Dynamic Response",
    xaxis2_title="Time (s)",
    height=600
)
fig_tl.show()


In [ ]:
# 2. State Ribbon / Gantt Timeline Visualization
fig_states = px.timeline(
    df_sim.with_columns(
        start=pl.col("time_s"),
        finish=pl.col("time_s") + 0.1,
        Task=pl.lit("Behaviour State")
    ).to_pandas(),
    x_start="start", x_end="finish", y="Task", color="state",
    title="Behaviour State Machine Transitions Over Time",
    color_discrete_map={
        "FOLLOW_LANE": "#2ecc71",
        "DECELERATE_TO_STOP": "#f39c12",
        "STAY_STOPPED": "#e74c3c",
        "CROSS_INTERSECTION": "#3498db"
    }
)
fig_states.update_layout(height=300, xaxis_title="Time (seconds)")
fig_states.show()


## 4. Verification Tests

In [ ]:
# Verify state cycle coverage
unique_states = df_sim["state"].unique().to_list()
assert "FOLLOW_LANE" in unique_states
assert "DECELERATE_TO_STOP" in unique_states
assert "STAY_STOPPED" in unique_states
assert "CROSS_INTERSECTION" in unique_states

# Verify complete stop
min_speed = df_sim.filter(pl.col("state") == "STAY_STOPPED")["ego_v_mps"].min()
assert min_speed < 0.2, f"Vehicle did not achieve full stop! Min speed was {min_speed}"

print("✅ All Behaviour State Machine tests passed successfully!")
